In [1]:
"""
配置常量模块
"""
from pathlib import Path
import torch

# ================= 配置区域 =================
# 输入数据的根目录
INPUT_ROOT = Path("/data/train/train_30/2025-08-19_19-35-52")

# OBJ 模型文件路径
OBJ_PATH = "/home/viplab/liukejia/project/depth_model/make_data/Houston_blender.obj"

# 渲染分辨率
H, W = 256, 256

# 设备选择
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ===========================================

In [2]:
"""
输入输出工具，深度及法线的保存与归一化
"""
import numpy as np
from PIL import Image
import torch

def save_results(results, output_dir, prefix):
    """
    保存: 
    1. NPY (深度/法线)
    2. JPG (可视化) 
    3. 打印深度统计
    """
    raw_depth = results['depth']
    mask = results['mask'] > 0.5
    
    # 打印有效区域深度 min/max
    if mask.sum() > 0:
        valid_depths = raw_depth[mask]
        d_min = valid_depths.min().item()
        d_max = valid_depths.max().item()
        print(f"[{prefix}] Depth Min: {d_min:.4f}, Max: {d_max:.4f}")
    else:
        print(f"[{prefix}] Depth Min/Max: N/A (No Object)")

    far = 300.0
    near = 0.1
    depth_norm = (far - raw_depth) / (far - near)
    depth_norm = torch.clamp(depth_norm, 0.0, 1.0)
    depth_norm[~mask] = 0.0
    npy_data = {
        "depth": depth_norm.cpu().numpy(),
        "normal_camera": results['normal_camera'].cpu().numpy()
    }
    np.save(output_dir / f"{prefix}_depth.npy", npy_data['depth'])
    np.save(output_dir / f"{prefix}_normal_cam.npy", npy_data['normal_camera'])
    # JPG 保存（uint8）
    depth_uint8 = (npy_data['depth'] * 255).astype(np.uint8)
    Image.fromarray(depth_uint8).convert("L").save(output_dir / f"{prefix}_depth.jpg", quality=95)
    nc_uint8 = (npy_data['normal_camera'] * 255).astype(np.uint8)
    Image.fromarray(nc_uint8).save(output_dir / f"{prefix}_normal_cam.jpg", quality=95)

In [3]:
from renderer import SimpleRenderer
from camera_utils import parse_json_camera
import math
from tqdm.notebook import tqdm 

renderer = SimpleRenderer(DEVICE)
renderer.load_mesh(OBJ_PATH)

json_files = list(INPUT_ROOT.rglob("*.json"))
print(f"Found {len(json_files)} JSON files.")

for json_path in tqdm(json_files, desc="Rendering"):
    try:
        # 直接保存在 json 同级目录
        output_dir = json_path.parent
        prefix = json_path.stem
        
        c2w, fovx = parse_json_camera(json_path, DEVICE)
        fovy = 2 * math.atan(math.tan(fovx / 2) * (W / H)) # aspect = W/H here to be safe
        
        res = renderer.forward(c2w, fovy, H, W)
        save_results(res, output_dir, prefix)
        
    except Exception as e:
        print(f"Error {json_path}: {e}")

print("All done.")

[Info] Loading mesh: /home/viplab/liukejia/project/depth_model/make_data/Houston_blender.obj ...
[Info] Loaded. V: 5104063, F: 8202718
Found 1 JSON files.


/home/viplab/anaconda3/envs/rd/lib/python3.10/site-packages/pytorch3d/io/obj_io.py:551: UserWarning: Mtl file does not exist: /home/viplab/liukejia/project/depth_model/make_data/Houston_blender.mtl
  warnings.warn(f"Mtl file does not exist: {f}")


Rendering:   0%|          | 0/1 [00:00<?, ?it/s]

[camconf] Depth Min: 133.0688, Max: 289.2617
All done.


In [4]:
# def clean_generated_files(root_dir: Path):
#     print(f"Cleaning in: {root_dir}")
#     # 更新为匹配 jpg
#     patterns = [
#         "*_depth.npy", 
#         "*_depth.jpg", 
#         "*_normal_cam.npy", 
#         "*_normal_cam.jpg"
#     ]
#     files = []
#     for p in patterns: files.extend(list(root_dir.rglob(p)))
#     files = sorted(list(set(files)))
    
#     if not files: return print("No files to clean.")
    
#     print(f"Found {len(files)} files.")
#     if input("Delete files? (y/n): ").lower() == 'y':
#         for f in tqdm(files, desc="Deleting"):
#             try: f.unlink()
#             except: pass
#         print("Cleaned.")
#     else:
#         print("Cancelled.")

# # clean_generated_files(INPUT_ROOT)